<a href="https://colab.research.google.com/github/carthomp99/ML-SARShipDetection-Tutorial/blob/main/ML-SARShipDetection-Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAR Ship Detection - YOLO and RT-DETR Comparison Tutorial
##### __by Carter Thompson__
##### updated: 2 May 2026
The goal of this tutorial is to show how to train, utilize, and compare the efficacy of two machine learning models for SAR Ship Detection. YOLO models have become mainstays in the digital image processing boom through the mid-2010s to the present; they are quick and easy to implement, but have sometimes lacked in accuracy (though newer models continue to improve on this). DETR transformer models are very robust and precise, but have lacked the speeds YOLO models could achieve - until now, with the development of RT-DETR models that operate in real time.

This tutorial will show you how to prepare, train, optimize, and test both of these models separately, and provide you tools to compare their performance metrics against one another if that interests you.

# References
Datasets, Github Repos, YT Vids, etc

[1] Zhang, T.; Zhang, X.; Li, J.; Xu, X.; Wang, B.; Zhan, X.; Xu, Y.; Ke, X.; Zeng, T.; Su, H.; et al. SAR Ship Detection Dataset (SSDD): Official Release and Comprehensive Data Analysis. Remote Sens. 2021, 13, 3690. https://doi.org/10.3390/rs13183690

[2] G. Jocher and J. Qiu, Ultralytics YOLO26. 2026. [Online]. Available: https://github.com/ultralytics/ultralytics

[3] W. Lv et al., “DETRs Beat YOLOs on Real-time Object Detection.” 2023.

# Top Level Library Requirements
Provide a list of required libraries and how to install them. Make
sure to include appropriate versions of each library. Use `assert`
commands to ensure that the libraries versions are available.

Provide the installation code near the top of each tutorial, below
the scope. Give users the option to skip installations if running
the code for a second time. You can do this with a simple `if`
statement. You also may want to add `pip` commands for installing
different libraries.

In [2]:
import time
import datetime
import os
import json
from collections import defaultdict

# Data Manipulation
import numpy as np

# Plotting/Visualization
import matplotlib.pyplot as plt

# ML Libraries
import torch
import torchvision
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold

# Ultralytics install
try:
  from ultralytics import YOLO
  from ultralytics import RTDETR
except ImportError:
  %pip install -q ultralytics

# Dataset and Model Gets from GitHub
if not os.path.exists("SSDD.zip"):
  !wget https://github.com/carthomp99/ML-SARShipDetection-Tutorial/raw/main/SSDD.zip
  !unzip -q ./SSDD.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.8 MB/s eta 0:00:00
--2026-05-02 21:49:47--  https://github.com/carthomp99/ML-SARShipDetection-Tutorial/raw/main/SSDD.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/carthomp99/ML-SARShipDetection-Tutorial/main/SSDD.zip [following]
--2026-05-02 21:49:47--  https://raw.githubusercontent.com/carthomp99/ML-SARShipDetection-Tutorial/main/SSDD.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61014353 (58M) [application/zip]
Saving to: ‘SSDD.zip’

SSDD.zip            100%[===================>]  58.19M   315MB/s    in 0.2s    

2026-05-02 

# Google Drive Mount
You can mount your Google Drive to this Colab runtime using [method].
Describe how to do this.


# Dataset Requirements
For the purposes of this tutorial I train, validate, and test my models on subsets of the SAR Ship Detection Dataset [1] using fivefold cross validation. A prepared version of this dataset specifically for this tutorial can be found on my GitHub: [github link] and was already imported above.

This dataset is composed of 1,160 SAR images, taken both at shore and offshore on open waters. The training set contains 928 images, and the testing set contains 232 images (80-20 split). The size on disk of the full dataset, including annotation information, is approximately 64.2 MB.

To access this dataset for use in this project or others, [tell them how to use wget correctly and refer up to the code snippet that will do this]. Downloads are also available [at the places they are available]

To use your own dataset, [tell how to change the wget to access a different set of data or how to upload files to the runtime]. Using [whatever the function names end up being] will allow you to prepare a test set with five equally-sized folds.

# Hardware
YOLO models are optimized to run well on both CPU and GPU hardware. If you are only interested in the YOLO portions of this tutorial, feel free to use the CPU runtime listed in Colab. For the sake of the RT-DETR portions and the comparisons between both models, this tutorial assumes usage of the free T4 GPU runtimes available through Colab. Ensure you have selected this option under `Runtime > Change runtime type` before attempting to run any of the code blocks listed below.

Note: If you ran the library steps before switching runtimes, ensure you run them again in the T4 GPU runtime or you will encounter errors later on.

# Approximate Execution Times

Approximate execution times were estimated based on the duration of each code block running in the Colab T4 GPU runtime. Results may vary.
- Dataset setup:
- Load/Save YOLO:
- Load/Save DETR:
- Optimize YOLO:
- Optimize DETR:
- Test YOLO:
- Test DETR:
- Fine-Tune YOLO:
- Fine-Tune DETR:
- Full Train YOLO:
- Full Train DETR:
- Comparison and Analysis:

# Expected Results (metrics)

# \#1. Dataset Tutorial

Provide sufficient information to define all elements of a dataset:

-   Explain how to setup a dataset using Google Drive. Explain the
    directory structure and provide any zip, unzip commands.
-   Provide code to play audio, display image, or play video based
    on what you are doing.
-   Describe what the inputs are. Is it 1D/2D/3D NumPy array?
-   Describe what the output labels are: What are the categories?
-   Describe how to extend the dataset. Show how to download and add
    an audio, image, or a plain video to the dataset.
-   Explain how to setup the training, validation, and testing
    datasets.

## Annotation Structure
The SSDD comes with COCO-style .json annotations. Ultralytics expects the following structure for its models: a single .txt file per image with each line in the file taking the form

`<class_id> <x_center> <y_center> <width> <height>`

There is only a single class_id for this tutorial as we are __only__ concerned with identifying ships among any and all other image data.

The following code converts COCO to the required form. For this tutorial I will run it on both the test and train sets of the SSDD. To run on your own data, simply modify the directory paths in the second block below.

In [4]:
def convert_coco(json_path, images_dir, labels_dir):
  os.makedirs(labels_dir, exist_ok=True)

  # ===== LOAD JSON =====
  with open(json_path, "r") as f:
      data = json.load(f)

  # ===== MAP IMAGE ID → IMAGE INFO =====
  images = {img["id"]: img for img in data["images"]}

  # ===== GROUP ANNOTATIONS BY IMAGE =====
  annots_by_image = defaultdict(list)
  for ann in data["annotations"]:
      annots_by_image[ann["image_id"]].append(ann)

  # ===== CONVERT =====
  missing_images = 0
  written_files = 0

  for image_id, img_info in images.items():
      file_name = img_info["file_name"]
      w, h = img_info["width"], img_info["height"]

      # IMPORTANT: adjust path if needed
      img_path = os.path.join(images_dir, file_name)

      if not os.path.exists(img_path):
          missing_images += 1
          continue

      label_path = os.path.join(
          labels_dir,
          file_name.replace(".jpg", ".txt")
      )

      lines = []

      for ann in annots_by_image.get(image_id, []):
          x, y, bw, bh = ann["bbox"]

          # COCO → YOLO normalization
          x_center = (x + bw / 2) / w
          y_center = (y + bh / 2) / h
          bw_norm = bw / w
          bh_norm = bh / h

          cls = ann["category_id"]

          lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}")

      # write only if annotations exist
      if lines:
          with open(label_path, "w") as f:
              f.write("\n".join(lines))
          written_files += 1

  # ===== SUMMARY =====
  print("Done.")
  print(f"Labels written: {written_files}")
  print(f"Missing images skipped: {missing_images}")

Done.
Labels written: 232
Missing images skipped: 0
Done.
Labels written: 928
Missing images skipped: 0


In [ ]:
# Test set conversion
json_path = "./SSDD/annotations/test.json"
images_dir = "./SSDD/images/test"
labels_dir = "./SSDD/labels/test"

convert_coco(json_path, images_dir, labels_dir)

# Train set conversion
json_path = "./SSDD/annotations/train.json"
images_dir = "./SSDD/images/train"
labels_dir = "./SSDD/labels/train"

convert_coco(json_path, images_dir, labels_dir)

## Dataset Assembly

With labels generated appropriately, we now need to structure the dataset to perform five-fold cross validation.

# \#2. Model Description
Provide sufficient information to access all elements of the model:

Load a model:   
Give Python code of how to load a pretrained model.
You may want to use `wget` from a GitHub account.

Save a model:   
Give Python code of how to save your model.

Number of parameters:   
You can give a command that prints how many.

Input layer:   
Demonstrate compatibility with the dataset tutorial.

Output layer:   
How many outputs? What are the activation functions? How do we
    train the model? What is the loss function? Do you use
    cross-entropy, mean squared error? What is it?

Intermediate layers:   
Is the model using CNNs? Is it LSTM? Is it ResNet? Give a brief
    description of intermediate layers. You can reuse images from
    the reference papers. If small, print the entire model. Else,
    describe major parts of the model.

## \#2a. YOLO Model

### Loading
To load a pretrained YOLO model, ensure first that the YOLO class is properly imported from ultralytics libraries, then use the following code:

In [3]:
from ultralytics import YOLO

# Load an existing YOLO model in for use, then display its information
model = YOLO("yolo26n.pt")
model.info()

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs


(260, 2572280, 0, 6.1192448)

You can replace `"yolo26n.pt" with any PyTorch (.pt extension) YOLO model, including [my-model-name.pt] provided at [the github link where my trained model will sit] to load in your own models for testing or other modification.

### Saving
You can export any Ultralytics model using the .export() method.. The default export format is a PyTorch file, but export() can also generate ONNX, TensorRT, and other common model formats. Further information on model exporting can be found in the Ultralytics documentation here [<--- PUT THE LINK].

In [4]:
# Export trained model to default PyTorch format
model.export()

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 285ms
Prepared 4 packages in 1.47s
Installed 4 packages in 476ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 5.1s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (5.8s)
Results saved to /content/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26n.onnx'

### Input Layer

### Output Layer

### Intermediate Layers

## \#2b. RT-DETR Model

Ultralytics provides pretrained RT-DETR models within their Python API. These models can then be trained and subsequently saved for further use.

### Loading
To load a pretrained RT-DETR model, ensure first that the RTDETR class is properly imported from ultralytics libraries, then use the following code:

In [5]:
from ultralytics import RTDETR

# Load an existing RT-DETR model in for use, then display its information
model = RTDETR("rtdetr-l.pt")
model.info()

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs


(449, 32970476, 0, 108.3437056)

You can replace `"rtdetr-l.pt" with any PyTorch (.pt extension) RT-DETR model, including [my-model-name.pt] provided at [the github link where my trained model will sit] to load in your own models for testing or other modification.

### Saving
You can export any Ultralytics model using the .export() method.. The default export format is a PyTorch file, but export() can also generate ONNX, TensorRT, and other common model formats. Further information on model exporting can be found in the Ultralytics documentation here [<--- PUT THE LINK].

In [6]:
# Export trained model to the default PyTorch format
model.export()

WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 1
os.environ['CUDA_VISIBLE_DEVICES']: 


### Input Layer


### Output Layer

### Intermediate Layers

# \#3. Model Optimization

Provide sufficient information to show how to fine-tune a model:

Loss function:
Specify the loss function (as in the model description).

Optimization algorithm:   
Give the name. Is it Adams? Which one.
Give basic parameters of the optimization algorithm.

Learning rate:   
Give initial learning rate.
If you adjust it, explain how.

Batch size:   
Give the number of audio, images, or videos that are used in
    each batch.

Number of epochs:   
Provide the maximum number of epochs. Are you using early
    stopping?

Train versus validation loss graphs:   
Explain any gaps between the training and validation graphs. Did
    you use early stopping? How did you avoid overfitting?

## \#3a. YOLO Optimization

## \#3b. RT-DETR Optimization
To optimize this RT-DETR model for our SAR application, we need to do the following:
* Create dataset YAML
* Train

# \#4. Basic Testing Tutorial
Load the pre-trained model using wget to download it and run it on
one test example.

## \#4a. Testing the YOLO Model

## \#4b. Testing the DETR Model

# \#5. Basic Fine-Tuning
Download a pre-trained model, setup the optimization loop, and show
that the loss function is getting reduced. Retrain the pre-trained
model for a few iterations to demonstrate the optimization process.

## \#5a. Fine-Tuning the YOLO Model

## \#5b. Fine-Tuning the RT-DETR Model

# \#6. Full Training
Run the loop multiple times and show that the validation and testing
losses converge to a good number. If the losses do not converge, you
will need to add data augmentation.

## \#6a. Top-to-bottom YOLO Training

## \#6b. Top-to-bottom DETR Training

# \#7. Comparison and Analysis Tutorial
Where I will give a tutorial on how to take the outputs and meaningfully compare them to understand the differences/improvements